# Layer 3 Training — Google Colab

**Runtime:** Runtime → Change runtime type → **T4 GPU** (required for sentence-transformers; Vosk runs on CPU regardless).

**Run cells top-to-bottom.** Colab disconnects mid-run — that's fine. The prep step is resume-safe and data is persisted to Google Drive. Reconnect, re-run cells 1–3 to restore the environment, then re-run whichever prep/train cell was interrupted.

---
**Time estimates:**
- Setup (cells 1–3): ~5 min
- Dataset download (cell 4): ~5 min
- Data prep / ASR (cell 5): **several hours** — resume-safe, can be done across multiple sessions
- Training (cell 6): ~10–15 min
- Evaluation (cell 7): ~2 min

In [ ]:
# ── Cell 1: Mount Google Drive (persists data across reconnections) ──────────
from google.colab import drive
drive.mount('/content/drive')

# All heavy data lives here — survives Colab restarts
DRIVE_DIR = '/content/drive/MyDrive/vibhu_capstone'

import os
os.makedirs(f'{DRIVE_DIR}/data/tele_antifraud', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
print('Drive mounted at', DRIVE_DIR)

In [ ]:
# ── Cell 2: Clone repo + install deps ───────────────────────────────────────
import os

REPO_URL = 'https://github.com/REPLACE_WITH_YOUR_REPO_URL/vibhu-capstone.git'

if not os.path.exists('/content/vibhu-capstone'):
    !git clone {REPO_URL} /content/vibhu-capstone
else:
    print('Repo already cloned — pulling latest')
    !git -C /content/vibhu-capstone pull

%cd /content/vibhu-capstone

!pip install -q -r requirements.txt
!pip install -q -r requirements_layer2.txt
!pip install -q -r requirements_layer3.txt
print('Deps installed')

In [ ]:
# ── Cell 3: Vosk ASR model ───────────────────────────────────────────────────
# Symlink Drive models dir into repo so artifacts persist across reconnections
import os

DRIVE_DIR = '/content/drive/MyDrive/vibhu_capstone'

# Symlink models/ -> Drive
if not os.path.islink('/content/vibhu-capstone/models'):
    if os.path.exists('/content/vibhu-capstone/models'):
        # Copy any files already there (Vosk bundled model) then replace with symlink
        !cp -r /content/vibhu-capstone/models/* {DRIVE_DIR}/models/ 2>/dev/null || true
        !rm -rf /content/vibhu-capstone/models
    !ln -s {DRIVE_DIR}/models /content/vibhu-capstone/models
    print('models/ -> Drive symlinked')
else:
    print('models/ already symlinked')

# Symlink data/tele_antifraud/ -> Drive
os.makedirs('/content/vibhu-capstone/data', exist_ok=True)
if not os.path.islink('/content/vibhu-capstone/data/tele_antifraud'):
    !ln -s {DRIVE_DIR}/data/tele_antifraud /content/vibhu-capstone/data/tele_antifraud
    print('data/tele_antifraud/ -> Drive symlinked')
else:
    print('data/tele_antifraud/ already symlinked')

# Download Vosk model if not already on Drive
vosk_dir = f'{DRIVE_DIR}/models/vosk-model-small-en-us-0.15'
if not os.path.exists(vosk_dir):
    print('Downloading Vosk model...')
    !wget -q -P {DRIVE_DIR}/models https://alphacephei.com/vosk/models/vosk-model-small-en-us-0.15.zip
    !unzip -q {DRIVE_DIR}/models/vosk-model-small-en-us-0.15.zip -d {DRIVE_DIR}/models/
    !rm {DRIVE_DIR}/models/vosk-model-small-en-us-0.15.zip
    print('Vosk model ready')
else:
    print('Vosk model already on Drive')

In [ ]:
# ── Cell 4: Download TeleAntiFraud-28k dataset ───────────────────────────────
# Cached to Drive via symlink. Safe to re-run — skips if already downloaded.
%cd /content/vibhu-capstone
!python training/download_dataset.py

In [ ]:
# ── Cell 5: Prepare data (L1 + L2 over each audio file) ──────────────────────
# This runs Vosk ASR on all 28k clips. Takes several hours total.
# RESUME-SAFE: already-processed files are skipped. Re-run freely after disconnects.
# Progress is saved to Google Drive as each file completes.
%cd /content/vibhu-capstone
!python training/prepare_data.py --split train
!python training/prepare_data.py --split validation
!python training/prepare_data.py --split test

In [ ]:
# ── Cell 6: Train all three classifiers ──────────────────────────────────────
# text: ~5 min (sentence-transformer embedding + LogReg)
# acoustic: ~3 min (XGBoost on 39-dim features)
# fusion: ~2 min (meta-classifier on val split)
# Models are saved to Drive via symlink.
%cd /content/vibhu-capstone
!python training/train_text.py
!python training/train_acoustic.py
!python training/train_fusion.py

In [ ]:
# ── Cell 7: Evaluate on held-out test split ───────────────────────────────────
%cd /content/vibhu-capstone
!python training/evaluate.py

# Show confusion matrix inline
from IPython.display import Image
Image('reports/confusion_matrix.png')

In [ ]:
# ── Cell 8: Print ablation table ─────────────────────────────────────────────
import json
ablation = json.load(open('reports/ablation.json'))
print('\n=== ABLATION TABLE ===')
print(f"  Text-only  macro-F1 : {ablation['text_only_macro_f1']:.4f}")
print(f"  Acoustic   macro-F1 : {ablation['acoustic_only_macro_f1']:.4f}")
print(f"  Fused      macro-F1 : {ablation['fused_macro_f1']:.4f}")
winner = max(ablation, key=ablation.get).replace('_macro_f1','').replace('_',' ')
print(f"\n  Best: {winner}")

In [ ]:
# ── Cell 9: Download trained artifacts ───────────────────────────────────────
# Zips models + reports and triggers a browser download.
# Unzip into your local repo root: unzip trained_models.zip
%cd /content/vibhu-capstone
!zip -r trained_models.zip \
    models/text_classifier.joblib \
    models/text_vocab.json \
    models/acoustic_classifier.joblib \
    models/fusion_classifier.joblib \
    models/acoustic_importance.json \
    reports/

from google.colab import files
files.download('trained_models.zip')